[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/08_parameter_plausibility_and_constrained_reruns.ipynb)

# Step 08 - Parameter plausibility and constrained reruns

This notebook runs the local Step 08 pipeline without Google Drive dependencies and writes auditable outputs under `outputs/parameter_plausibility/`.

**Scope:** reviewer-facing plausibility checks use all accepted cells by default with one best accepted candidate per cell, and constrained projections run for that full cell-level set. The editable parameter range table is reused from `outputs/parameter_plausibility/parameter_ranges.csv` when present.

**Claim scope:** Step 08 audits whether accepted parameters are plausible and interpretable. It does not authorize final biological degeneracy wording because Step 09 synthesis is still pending.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
PROJECT_ROOT

In [ ]:
from src.step08_parameter_plausibility import (
    Step08Config,
    compare_step08_runtime_presets,
    run_step08_parameter_plausibility,
)

config = Step08Config(
    max_candidates=None,
    candidate_policy="best_per_cell",
    constrained_max_candidates=None,
    write_outputs=True,
)
result = run_step08_parameter_plausibility(PROJECT_ROOT, config)
result["analysis_summary"]

## 1. Accepted ensemble inventory

The Step 08 input contract preserves the same cell/candidate identity, brain region, condition, held-out metrics, and mechanism labels used by Steps 05–07.

In [ ]:
inventory_cols = [
    "file_id", "region", "condition", "candidate_id", "mechanism_cluster", "dominant_mechanism",
    "holdout_mean_rmse_mV", "holdout_mean_pass_fraction",
]
inventory = result["candidates"][inventory_cols].copy()
inventory

## 2. Parameter plausibility and identifiability audit

Each candidate x parameter row receives a plausibility status, an identifiability status from Step 03 where available, explicit lower/upper-bound flags, and an interpretation guardrail. `above_upper_bound=False` means the fitted value is not above the declared broad upper bound; it does not mean the parameter is identifiable or biologically interpretable.

In [ ]:
audit = result["parameter_range_audit"]
display_cols = [
    "file_id", "region", "condition", "candidate_id", "parameter", "coordinate_type", "value",
    "lower_bound", "upper_bound", "below_lower_bound", "above_upper_bound", "bound_violation",
    "plausibility_status", "identifiability_status", "physiologically_interpretable", "interpretation_guardrail",
]
audit[display_cols].head(30)

In [ ]:
status_counts = (
    audit.groupby(["coordinate_type", "plausibility_status", "identifiability_status"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
)
status_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_data = audit.groupby(["parameter", "plausibility_status"]).size().unstack(fill_value=0)
plot_data.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("candidate-parameter rows")
ax.set_title("Step 08 plausibility status by parameter")
ax.legend(title="plausibility", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig

## 3. Effective-parameter plausibility

Effective coordinates are exported separately because they are the preferred reviewer-facing representation for structurally confounded raw factors.

In [ ]:
effective = result["effective_parameter_plausibility"]
effective.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
eff_plot = effective.groupby(["parameter", "physiologically_interpretable"]).size().unstack(fill_value=0)
eff_plot.plot(kind="bar", stacked=True, ax=ax, color=["#d95f02", "#1b9e77"])
ax.set_ylabel("candidate-parameter rows")
ax.set_title("Effective-coordinate interpretability")
ax.legend(title="interpretable", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig

## 4. Constrained rerun comparison

The comparison is `step04_unconstrained_candidate_vs_broad_range_projection`: the unconstrained side is the accepted Step 04 candidate, and the constrained side is a lightweight projection into broad plausibility ranges followed by resimulation. This is not a new optimizer and cannot replace Step 04; it tests whether the current claim is sensitive to broad parameter guardrails.

In [ ]:
constrained = result["constrained_rerun_comparison"]
constrained

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
if not constrained.empty:
    x = range(len(constrained))
    ax.plot(x, constrained["unconstrained_holdout_rmse_mV"], marker="o", label="unconstrained")
    ax.plot(x, constrained["constrained_holdout_rmse_mV"], marker="s", label="constrained screen")
    ax.set_xticks(list(x))
    ax.set_xticklabels(constrained["candidate_id"], rotation=45, ha="right")
ax.set_ylabel("held-out RMSE (mV)")
ax.set_title("Unconstrained vs constrained screen")
ax.legend()
fig.tight_layout()
fig

## 5. Candidate-level interpretability status

Candidate-level rows combine plausibility, identifiability, and constrained-screen persistence into conservative claim statuses.

In [ ]:
status = result["interpretability_status"]
status

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
claim_counts = status["parameter_interpretability_status"].value_counts().sort_index()
claim_counts.plot(kind="barh", ax=ax)
ax.set_xlabel("candidate count")
ax.set_title("Step 08 candidate-level parameter claim status")
fig.tight_layout()
fig

## 6. Performance tuning

A small runtime comparison documents whether the default notebook setting remains practical.

In [ ]:
performance = compare_step08_runtime_presets(PROJECT_ROOT, max_candidates=1)
output_dir = PROJECT_ROOT / "outputs" / "parameter_plausibility"
output_dir.mkdir(parents=True, exist_ok=True)
performance.to_csv(output_dir / "performance_benchmark.csv", index=False)
performance

## Conservative conclusion

Step 08 permits cautious statements about which accepted coordinates are within broad plausibility ranges and which are interpretable only as effective parameters. Final biological degeneracy claims remain disabled until Step 09 integrates provenance, assumptions, predictions, parameter plausibility, and manuscript-facing tables.

In [ ]:
summary_path = PROJECT_ROOT / "outputs" / "parameter_plausibility" / "analysis_summary.json"
print(summary_path)
print(result["analysis_summary"]["claim_scope"])

In [ ]:
assert {"below_lower_bound", "above_upper_bound", "bound_violation"}.issubset(audit.columns)
assert constrained["comparison_kind"].eq("step04_unconstrained_candidate_vs_broad_range_projection").all()
assert not status["final_degeneracy_claim_allowed_after_step08"].astype(bool).any()
print("Step 08 full-cell plausibility notebook completed across the full cell-level target scope.")

## Post-execution scientific status

Executed status for reviewer response: Step 08 evaluated 30 best-per-cell candidates across 270 candidate-parameter rows and 180 constrained current rows. All 30 candidates are downgraded for out-of-range raw parameters, even though broad-constrained prediction/mechanism persistence is frequent. This supports R4 guardrails: effective coordinates and range flags can be reported, but raw physiological parameter interpretation and final biological degeneracy claims remain blocked.